# Day 037 — Exercise 4: deduplicate

**What you'll build:** `deduplicate(df, subset=None) -> pd.DataFrame` — remove duplicate rows with `df.drop_duplicates()` and reset the index. The optional `subset` list restricts which columns define a duplicate.

**Why it matters:** Double-processing an API response, re-running an ETL job, or concatenating overlapping exports all produce duplicates. `drop_duplicates` keeps the first occurrence by default — an auditable, deterministic choice.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import io
import pandas as pd

import pandas as pd

def drop_or_fill_nulls(df: pd.DataFrame, strategy: str = 'mean') -> pd.DataFrame:
    result   = df.copy()
    if strategy == 'drop':
        return result.dropna().reset_index(drop=True)
    num_cols = result.select_dtypes(include='number').columns
    if strategy == 'zero':
        result[num_cols] = result[num_cols].fillna(0)
    elif strategy == 'mean':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].mean())
    elif strategy == 'median':
        for col in num_cols:
            result[col] = result[col].fillna(result[col].median())
    else:
        raise ValueError(
            f"Unknown strategy {strategy!r}. "
            "Use 'drop', 'zero', 'mean', or 'median'."
        )
    return result


import pandas as pd

def coerce_numeric_columns(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    result = df.copy()
    for col in columns:
        result[col] = pd.to_numeric(result[col], errors='coerce')
    return result


import pandas as pd

def clean_string_column(df: pd.DataFrame, col: str) -> pd.DataFrame:
    result = df.copy()
    result[col] = result[col].str.strip().str.lower()
    return result


# DUPE_DF: Widget(25) appears at rows 0 and 2; Gadget(150) at rows 1 and 4
DUPE_CSV = (
    'product,price\n'
    'Widget,25\n'
    'Gadget,150\n'
    'Widget,25\n'
    'Doohickey,8\n'
    'Gadget,150'
)
DUPE_DF = pd.read_csv(io.StringIO(DUPE_CSV))

## Your Implementation

In [ ]:
def deduplicate(df: pd.DataFrame, subset: list | None = None) -> pd.DataFrame:
    """
    Remove duplicate rows and reset the index.

    Args:
        df     — source DataFrame
        subset — column name list to compare (None = all columns)
    Returns:
        Deduplicated DataFrame with index reset to 0, 1, 2, ...
    """
    # TODO: return df.drop_duplicates(subset=subset).reset_index(drop=True)
    pass

## Check Your Work

In [ ]:
def _run_checks():
    total = 5
    passed = 0

    # Check 1: defined, returns DataFrame
    try:
        assert 'deduplicate' in globals()
        result = deduplicate(DUPE_DF)
        assert isinstance(result, pd.DataFrame), \
            f'expected DataFrame, got {type(result).__name__}'
        passed += 1; print('\u2705 Check 1: returns a DataFrame')
    except Exception as e:
        print(f'\u274c Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: duplicate rows removed (5 → 3)
    try:
        result = deduplicate(DUPE_DF)
        assert len(result) == 3, f'expected 3 rows, got {len(result)}'
        products = set(result['product'])
        assert products == {'Widget', 'Gadget', 'Doohickey'}, \
            f'unexpected products: {products}'
        passed += 1; print('\u2705 Check 2: 5 rows → 3 unique rows')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: index is reset (0, 1, 2)
    try:
        result = deduplicate(DUPE_DF)
        assert list(result.index) == [0, 1, 2], \
            f'index not reset: {list(result.index)}'
        passed += 1; print('\u2705 Check 3: index reset to 0, 1, 2')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: subset parameter limits which columns define equality
    try:
        mixed = pd.DataFrame({
            'product': ['Widget', 'Widget', 'Gadget'],
            'price':   [25, 30, 150],
        })
        result_all = deduplicate(mixed)
        result_sub = deduplicate(mixed, subset=['product'])
        assert len(result_all) == 3, \
            f'all-columns dedup: expected 3 rows, got {len(result_all)}'
        assert len(result_sub) == 2, \
            f'subset dedup: expected 2 rows, got {len(result_sub)}'
        passed += 1; print('\u2705 Check 4: subset parameter works correctly')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: original DataFrame not mutated
    try:
        _ = deduplicate(DUPE_DF)
        assert len(DUPE_DF) == 5, \
            f'original changed: expected 5 rows, got {len(DUPE_DF)}'
        passed += 1; print('\u2705 Check 5: original DataFrame not mutated')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Exercise complete!')
    print(f'\nScore: {passed}/{total}')


_run_checks()

## Solution

<details>
<summary>Click to reveal</summary>

```python
import pandas as pd

def deduplicate(df: pd.DataFrame, subset: list | None = None) -> pd.DataFrame:
    return df.drop_duplicates(subset=subset).reset_index(drop=True)
```

</details>